# Spur / Turso flamegraph — quack-flamegraph + GGSQL

Live analysis of the folded CPU profile from Spur indexing Turso.

1. **Extract** with the local `quack_flamegraph` DuckDB extension (`flamegraph_coverage`, `flamegraph_edges`, `flamegraph_hot_stacks`).
2. **Visualize** hotspots and bottlenecks with GGSQL (bar, heatmap, Pareto line).

Source folded file: `test/data/spur-turso-demo.folded` (checked-in excerpt of the Spur/Turso profile).
SQL cells read parquet relative to this notebook: `../../../build/turso-analysis/`.
Unique-stack coverage is the ranking metric (not unnest-inclusive).

In [3]:
import subprocess, sys
from pathlib import Path


def repo_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "Cargo.toml").exists() and (p / "scripts" / "extract_folded_hotspots.py").exists():
            return p
    raise FileNotFoundError(f"repo root not found from cwd={here}")


ROOT = repo_root()
VENV_PY = ROOT / "configure/venv/bin/python"
SCRIPT = ROOT / "scripts/extract_folded_hotspots.py"
OUT = ROOT / "build/turso-analysis"

# Kernel duckdb is v1.5.4; quack_flamegraph is built for v1.5.5.
# Extract in the project venv, then GGSQL cells read parquet.
r = subprocess.run([str(VENV_PY), str(SCRIPT)], check=True, capture_output=True, text=True, cwd=ROOT)
print(r.stdout)
if r.stderr:
    print(r.stderr, file=sys.stderr)
print("parquet", sorted(p.name for p in OUT.glob("*.parquet")))

samples 10000 duckdb 1.5.5
wrote ['buckets.parquet', 'coverage.parquet', 'edges.parquet', 'exclusive.parquet', 'hot_stacks.parquet', 'pareto.parquet']

parquet ['buckets.parquet', 'coverage.parquet', 'edges.parquet', 'exclusive.parquet', 'hot_stacks.parquet', 'pareto.parquet']


## Unique-stack coverage (hotspots)

`flamegraph_coverage` counts each stack once. Rank is **not** unnest-inclusive — recursive names do not inflate.

In [5]:
SELECT label AS frame, coverage, n_stacks, pct
FROM read_parquet('../../../build/turso-analysis/coverage.parquet')
ORDER BY pct DESC

frame,coverage,n_stacks,pct
git_walk::run_full_walk_into,8840,9,88.4
git_walk::symbol_changes_for_commit,5540,5,55.4
extract::tree_sitter::BytesExtractor::extract,4900,4,49.0
git_walk::cached_extract,4900,4,49.0
extract::tree_sitter::run_query,4900,4,49.0
ts_node_parent,3900,3,39.0
ts_node_child_with_descendant,3770,2,37.7
ts_node_child_iterator_next,2470,1,24.7
build_full_artifact_for_graph_build,2150,2,21.5
ts_parser_parse,1190,1,11.9


## Exclusive leaf time vs disjoint buckets

Leaves lie under LTO. The bucket chart is mutually exclusive unique-stack coverage (priority `run_query` > `buckets` > `parse` > `capture_text` > `tokens` > git-walk > other).

In [8]:
SELECT label AS frame, exclusive, pct
FROM read_parquet('../../../build/turso-analysis/exclusive.parquet')
ORDER BY pct DESC

frame,exclusive,pct
ts_node_child_iterator_next,2470,24.7
ts_node_child_with_descendant,1300,13.0
ts_parser_parse,1190,11.9
other_work,1160,11.6
ts_query_cursor_next_match,1000,10.0
store::build::buckets_from_facts,960,9.6
extract::tree_sitter::contained_capture_text,700,7.0
git_walk::symbol_changes_for_commit,640,6.4
extract::collect_symbol_tokens,450,4.5
ts_node_parent,130,1.3


In [7]:
SELECT bucket, samples, n_stacks, pct
FROM read_parquet('../../../build/turso-analysis/buckets.parquet')
ORDER BY pct DESC

bucket,samples,n_stacks,pct
F_walk_git,6500,6,65.0
C_parse,1190,1,11.9
G_other,1160,1,11.6
E_capture_text,700,1,7.0
D_tokens,450,1,4.5


## Bottleneck edges (`flamegraph_edges`)

Parent→child sample mass. The hot tile should be `run_query` → `ts_node_parent` (~37% of all samples) — the string-literal parent walk, not QueryCursor.

In [4]:
SELECT parent, child, samples, pct
FROM read_parquet('../../../build/turso-analysis/edges.parquet')
ORDER BY pct DESC
LIMIT 12

parent,child,samples,pct
git_walk::symbol_changes_for_commit,git_walk::cached_extract,4900,49.0
git_walk::cached_extract,...ree_sitter::BytesExtractor::extract,4900,49.0
...ree_sitter::BytesExtractor::extract,extract::tree_sitter::run_query,4900,49.0
extract::tree_sitter::run_query,ts_node_parent,3900,39.0
ts_node_parent,ts_node_child_with_descendant,3770,37.7
ts_node_child_with_descendant,ts_node_child_iterator_next,2470,24.7
git_walk::run_full_walk_into,build_full_artifact_for_graph_build,2150,21.5
ts_parser_parse_with_options,ts_parser_parse,1190,11.9
build_full_artifact_for_graph_build,extract::tree_sitter::build_facts,1190,11.9
extract::tree_sitter::extract_files,...er::extract_file_contents_from_tree,1190,11.9


In [10]:
SELECT parent || ' → ' || child AS edge, samples, pct
FROM read_parquet('../../../build/turso-analysis/edges.parquet')
ORDER BY pct DESC
LIMIT 10

edge,samples,pct
git_walk::symbol_changes_for_commit → git_walk::cached_extract,4900,49.0
git_walk::cached_extract → ...ree_sitter::BytesExtractor::extract,4900,49.0
...ree_sitter::BytesExtractor::extract → extract::tree_sitter::run_query,4900,49.0
extract::tree_sitter::run_query → ts_node_parent,3900,39.0
ts_node_parent → ts_node_child_with_descendant,3770,37.7
ts_node_child_with_descendant → ts_node_child_iterator_next,2470,24.7
git_walk::run_full_walk_into → build_full_artifact_for_graph_build,2150,21.5
ts_parser_parse_with_options → ts_parser_parse,1190,11.9
build_full_artifact_for_graph_build → extract::tree_sitter::build_facts,1190,11.9
extract::tree_sitter::extract_files → ...er::extract_file_contents_from_tree,1190,11.9


## Pareto of unique stacks + fattest stacks

16 stacks ≈ 50% of samples; the same `run_query` → `ts_node_parent` spine repeats. Fat stacks are `flamegraph_hot_stacks`.

In [9]:
SELECT rank, cum_pct, samples, leaf
FROM read_parquet('../../../build/turso-analysis/pareto.parquet')
WHERE rank <= 250
ORDER BY rank

rank,cum_pct,samples,leaf
1,24.7,2470,ts_node_child_iterator_next
2,37.7,1300,ts_node_child_with_descendant
3,49.6,1190,ts_parser_parse
4,61.2,1160,other_work
5,71.2,1000,ts_query_cursor_next_match
6,80.8,960,store::build::buckets_from_facts
7,87.8,700,extract::tree_sitter::contained_capture_text
8,94.2,640,git_walk::symbol_changes_for_commit
9,98.7,450,extract::collect_symbol_tokens
10,100.0,130,ts_node_parent


In [6]:
SELECT leaf, samples, depth, pct
FROM read_parquet('../../../build/turso-analysis/hot_stacks.parquet')
ORDER BY pct DESC

leaf,samples,depth,pct
ts_node_child_iterator_next,2470,9,24.7
ts_node_child_with_descendant,1300,8,13.0
ts_parser_parse,1190,8,11.9
other_work,1160,2,11.6
ts_query_cursor_next_match,1000,8,10.0
store::build::buckets_from_facts,960,5,9.6
extract::tree_sitter::contained_capture_text,700,3,7.0
git_walk::symbol_changes_for_commit,640,3,6.4
extract::collect_symbol_tokens,450,5,4.5
ts_node_parent,130,7,1.3


## Read the charts

| Rank | Bottleneck | Evidence | Action |
|---|---|---|---|
| 1 | `run_query` parent walk | Coverage ~49%; edge `run_query`→`ts_node_parent` ~37%; exclusive leaf `ts_node_child_iterator_next` | Stop unbounded `node.parent()` in `is_string_literal_context` |
| 2 | `buckets_from_facts` | Disjoint ~9.6%, mostly self-time leaf | Re-profile after #1 |
| — | `ts_parser_parse` | ~12% but C library, sequential phase before `run_query` | Do not start here |
| last | `collect_symbol_tokens` | Inclusive inflate; unique-stack ~4.5% | Ignore |

Fix one thing, re-run this notebook on a new folded file.